In [1]:
import json, random, subprocess
from pathlib import Path
from typing import Literal

import matplotlib.pyplot as plt
import ollama
import pandas as pd
from pydantic import BaseModel
from sklearn.metrics import confusion_matrix, precision_recall_fscore_support

Path("data").mkdir(exist_ok=True)
Path("results").mkdir(exist_ok=True)
THR = 50_000  # approval threshold in INR

In [2]:
rng = random.Random(42)
BASE = pd.Timestamp("2026-01-05")

ids = rng.sample(range(1000, 9999), 14)
buyers    = [f"U{i}" for i in ids[:6]]
approvers = [f"U{i}" for i in ids[6:10]]
payers    = [f"U{i}" for i in ids[10:14]]

VENDORS = ["Apex Office Supplies", "Bharat IT Hardware", "Calcutta Facility Services", "Delta Logistics",
           "Eastern Stationers", "Fortune Electricals", "Ganga Printing Works", "Horizon Software",
           "Indus Furniture", "Jupiter Cleaning Co", "Kalpana Catering", "Lotus Packaging"]
ITEMS = ["Office chairs", "Laptop batteries", "Printer toner", "Cleaning services", "Software licences",
         "Catering for event", "Network cables", "Packaging material", "Desk lamps", "Courier charges",
         "Printing of brochures", "Security cameras"]

rows, used_pairs = [], set()

def rand_date():
    return BASE + pd.Timedelta(days=rng.randint(0, 270))

def fresh_pair():
    """A (buyer, vendor) pair not used anywhere else, so the split-PO rule can't fire by accident."""
    while True:
        b, v = rng.choice(buyers), rng.choice(VENDORS)
        if (b, v) not in used_pairs:
            used_pairs.add((b, v))
            return b, v

def add(created, approved, paid, vendor, amount, date, vtype, hard=False):
    rows.append(dict(date=date, vendor=vendor, amount=round(amount, -1), created_by=created,
                     approved_by=approved, payment_processed_by=paid, description=rng.choice(ITEMS),
                     violation_type=vtype, hard_negative=hard))

low = lambda: rng.uniform(2_000, 39_900)  # ordinary amounts stay below 0.8 * threshold

# 68 ordinary clean POs: creator, approver, payer all different people
for _ in range(68):
    add(rng.choice(buyers), rng.choice(approvers), rng.choice(payers), rng.choice(VENDORS), low(), rand_date(), "none")
# 10 hard negatives: clean POs with amounts just under the threshold, but single and unrelated
for _ in range(10):
    b, v = fresh_pair()
    add(b, rng.choice(approvers), rng.choice(payers), v, rng.uniform(44_000, 49_500), rand_date(), "none", hard=True)
# 18 create_pay: same user created the PO and processed the payment
for _ in range(18):
    b = rng.choice(buyers)
    add(b, rng.choice(approvers), b, rng.choice(VENDORS), low(), rand_date(), "create_pay")
# 12 self_approve: same user created and approved
for _ in range(12):
    b = rng.choice(buyers)
    add(b, b, rng.choice(payers), rng.choice(VENDORS), low(), rand_date(), "self_approve")
# 12 split_po: 4 groups of 3 POs, same creator and vendor within 3 days, each just under the threshold
for _ in range(4):
    b, v = fresh_pair()
    d = rand_date()
    for k in range(3):
        add(b, rng.choice(approvers), rng.choice(payers), v, rng.uniform(44_000, 49_500),
            d + pd.Timedelta(days=k), "split_po")

df = pd.DataFrame(rows).sort_values("date").reset_index(drop=True)
df.insert(0, "po_id", [f"PO-{i+1:04d}" for i in range(len(df))])
df["violation"] = df.violation_type != "none"
df.to_csv("data/pos.csv", index=False)

print(df.violation_type.value_counts(), "\n")
display(df.head())

violation_type
none            78
create_pay      18
split_po        12
self_approve    12
Name: count, dtype: int64 



,po_id,date,vendor,amount,created_by,approved_by,payment_processed_by,description,violation_type,hard_negative,violation
0,PO-0001,2026-01-05,Fortune Electricals,30900.0,U4657,U9935,U1488,Courier charges,none,False,False
1,PO-0002,2026-01-06,Ganga Printing Works,7990.0,U5506,U2679,U2535,Security cameras,none,False,False
2,PO-0003,2026-01-07,Ganga Printing Works,11250.0,U3286,U7912,U4582,Laptop batteries,none,False,False
3,PO-0004,2026-01-12,Kalpana Catering,39870.0,U4657,U2679,U1520,Laptop batteries,none,False,False
4,PO-0005,2026-01-15,Delta Logistics,48980.0,U2824,U7912,U4582,Courier charges,none,True,False


In [3]:
def baseline(d):
    out = pd.Series("none", index=d.index)
    out[d.created_by == d.payment_processed_by] = "create_pay"
    out[d.created_by == d.approved_by] = "self_approve"
    near = d[(d.amount >= 0.8 * THR) & (d.amount < THR)]
    for i, r in near.iterrows():
        same = near[(near.created_by == r.created_by) & (near.vendor == r.vendor)
                    & ((near.date - r.date).abs() <= pd.Timedelta(days=7))]
        if len(same) >= 3:
            out[i] = "split_po"
    return out

df["base_type"] = baseline(df)
df["base_viol"] = df.base_type != "none"
display(pd.crosstab(df.violation_type, df.base_type))

def evaluate(d, pred_col):
    p, r, f, _ = precision_recall_fscore_support(d.violation, d[pred_col], average="binary", zero_division=0)
    per_type = {t: round(d[d.violation_type == t][pred_col].mean(), 2)
                for t in ["create_pay", "self_approve", "split_po"]}
    return dict(precision=round(p, 2), recall=round(r, 2), f1=round(f, 2),
                fp_on_clean=int(d[(~d.violation) & d[pred_col]].shape[0]),
                fp_on_hard_neg=int(d[d.hard_negative & d[pred_col]].shape[0]),
                recall_by_type=per_type)

print(evaluate(df, "base_viol"))

base_type,create_pay,none,self_approve,split_po
violation_type,,,,
create_pay,18,0,0,0
none,0,78,0,0
self_approve,0,0,12,0
split_po,0,0,0,12


{'precision': 1.0, 'recall': 1.0, 'f1': 1.0, 'fp_on_clean': 0, 'fp_on_hard_neg': 0, 'recall_by_type': {'create_pay': np.float64(1.0), 'self_approve': np.float64(1.0), 'split_po': np.float64(1.0)}}


In [4]:
class Verdict(BaseModel):
    violation: bool
    type: Literal["none", "create_pay", "self_approve"]

SYSTEM = """You are an internal-controls auditor reviewing ONE purchase order (PO) at a time.
Flag a segregation-of-duties violation only if one of these is true:
1. create_pay: the user ID in 'created by' is exactly the same as the user ID in 'payment processed by'.
2. self_approve: the user ID in 'created by' is exactly the same as the user ID in 'approved by'.
If the three user IDs are all different, the PO is fine: answer violation=false, type=none.
Compare the user IDs character by character. Reply with JSON only."""

def row_text(r):
    return (f"{r.po_id} | date {r.date:%Y-%m-%d} | vendor: {r.vendor} | amount: INR {r.amount:,.0f} | "
            f"created by: {r.created_by} | approved by: {r.approved_by} | payment processed by: {r.payment_processed_by}")

print(row_text(df.iloc[0]))

PO-0001 | date 2026-01-05 | vendor: Fortune Electricals | amount: INR 30,900 | created by: U4657 | approved by: U9935 | payment processed by: U1488


In [5]:
LLM_CACHE = Path("results/sod_llm.jsonl")
MODELS = ["qwen2.5:1.5b", "llama3.2:3b", "qwen2.5:3b"]

def classify(model, r):
    resp = ollama.chat(
        model=model,
        messages=[{"role": "system", "content": SYSTEM}, {"role": "user", "content": row_text(r)}],
        format=Verdict.model_json_schema(),
        options={"temperature": 0, "num_ctx": 2048, "num_predict": 60},
    )
    try:
        v = Verdict.model_validate_json(resp.message.content)
        ok, pv, pt = True, v.violation, v.type
    except Exception:
        ok, pv, pt = False, False, "none"
    return dict(model=model, po_id=r.po_id, parse_ok=ok, pred_viol=pv, pred_type=pt,
                secs=resp.total_duration / 1e9, prompt_tokens=resp.prompt_eval_count,
                gen_tokens=resp.eval_count, raw=resp.message.content)

# smoke test on 5 rows with the fastest model
for _, r in df.head(5).iterrows():
    o = classify("qwen2.5:1.5b", r)
    print(o["po_id"], "| truth:", r.violation_type, "| pred:", o["pred_type"], "|", round(o["secs"], 1), "s")

PO-0001 | truth: none | pred: self_approve | 26.1 s
PO-0002 | truth: none | pred: self_approve | 5.1 s
PO-0003 | truth: none | pred: self_approve | 5.5 s
PO-0004 | truth: none | pred: self_approve | 5.4 s
PO-0005 | truth: none | pred: self_approve | 5.2 s


In [6]:
def done_keys():
    if not LLM_CACHE.exists():
        return set()
    return {(j["model"], j["po_id"]) for j in map(json.loads, LLM_CACHE.read_text(encoding="utf-8").splitlines())}

done = done_keys()
for model in MODELS:
    print("=====", model)
    classify(model, df.iloc[0])  # warm-up, not logged; also primes the cached system prompt
    for _, r in df.iterrows():
        if (model, r.po_id) in done:
            continue
        o = classify(model, r)
        with LLM_CACHE.open("a", encoding="utf-8") as f:
            f.write(json.dumps(o) + "\n")
        if int(r.po_id[3:]) % 20 == 0:
            print(model, r.po_id, round(o["secs"], 1), "s")
    subprocess.run(["ollama", "stop", model], capture_output=True)

===== qwen2.5:1.5b
===== llama3.2:3b
===== qwen2.5:3b


In [7]:
res = pd.DataFrame(map(json.loads, LLM_CACHE.read_text(encoding="utf-8").splitlines()))
full = df.merge(res, on="po_id")

table = {"rule baseline": evaluate(df, "base_viol")}
for m in MODELS:
    d = full[full.model == m].copy()
    d["pred_viol"] = d.pred_viol.astype(bool)
    e = evaluate(d, "pred_viol")
    e["parse_fail_pct"] = round((~d.parse_ok).mean() * 100, 1)
    e["sec_per_row"] = round(d.secs.mean(), 1)
    table[m] = e
display(pd.DataFrame(table).T)

# the fairer view: only rows a single-row prompt can judge (drop the split_po rows)
rowlevel = {m: evaluate(full[(full.model == m) & (full.violation_type != "split_po")].assign(
                pred_viol=lambda x: x.pred_viol.astype(bool)), "pred_viol") for m in MODELS}
display(pd.DataFrame(rowlevel).T)

,precision,recall,f1,fp_on_clean,fp_on_hard_neg,recall_by_type,parse_fail_pct,sec_per_row
rule baseline,1.0,1.0,1.0,0,0,"{'create_pay': 1.0, 'self_approve': 1.0, 'spli...",NaN,NaN
qwen2.5:1.5b,0.35,1.0,0.52,78,10,"{'create_pay': 1.0, 'self_approve': 1.0, 'spli...",0.0,5.3
llama3.2:3b,0.0,0.0,0.0,0,0,"{'create_pay': 0.0, 'self_approve': 0.0, 'spli...",0.0,7.1
qwen2.5:3b,0.0,0.0,0.0,0,0,"{'create_pay': 0.0, 'self_approve': 0.0, 'spli...",0.0,6.3


,precision,recall,f1,fp_on_clean,fp_on_hard_neg,recall_by_type
qwen2.5:1.5b,0.28,1.0,0.43,78,10,"{'create_pay': 1.0, 'self_approve': 1.0, 'spli..."
llama3.2:3b,0.0,0.0,0.0,0,0,"{'create_pay': 0.0, 'self_approve': 0.0, 'spli..."
qwen2.5:3b,0.0,0.0,0.0,0,0,"{'create_pay': 0.0, 'self_approve': 0.0, 'spli..."


In [8]:
for m in MODELS:
    d = full[full.model == m]
    fn = d[(d.violation) & (~d.pred_viol.astype(bool)) & (d.violation_type != "split_po")]
    fp = d[(~d.violation) & d.pred_viol.astype(bool)]
    print(f"\n{m}: {len(fn)} missed row-level violations, {len(fp)} false alarms")
    display(fp[["po_id", "created_by", "approved_by", "payment_processed_by", "amount", "pred_type"]].head(8))


qwen2.5:1.5b: 0 missed row-level violations, 78 false alarms


,po_id,created_by,approved_by,payment_processed_by,amount,pred_type
0,PO-0001,U4657,U9935,U1488,30900.0,self_approve
3,PO-0002,U5506,U2679,U2535,7990.0,self_approve
6,PO-0003,U3286,U7912,U4582,11250.0,self_approve
9,PO-0004,U4657,U2679,U1520,39870.0,self_approve
12,PO-0005,U2824,U7912,U4582,48980.0,self_approve
21,PO-0008,U1409,U2424,U1488,7580.0,self_approve
36,PO-0013,U5012,U2424,U1488,14290.0,self_approve
39,PO-0014,U5012,U7912,U1488,13250.0,self_approve



llama3.2:3b: 30 missed row-level violations, 0 false alarms


,po_id,created_by,approved_by,payment_processed_by,amount,pred_type



qwen2.5:3b: 30 missed row-level violations, 0 false alarms


,po_id,created_by,approved_by,payment_processed_by,amount,pred_type


In [9]:
print(full.groupby(["model", "pred_type"]).size())
print(full[full.model == "llama3.2:3b"].raw.head(3).tolist())

df["always_yes"] = True
df["always_no"] = False
display(pd.DataFrame({"always_yes": evaluate(df, "always_yes"),
                      "always_no": evaluate(df, "always_no")}).T)

model         pred_type   
llama3.2:3b   none            120
qwen2.5:1.5b  self_approve    120
qwen2.5:3b    none            120
dtype: int64
['{"violation": false, "type": "none"}', '{"violation": false, "type": "none"}', '{"violation": false, "type": "none"}']


,precision,recall,f1,fp_on_clean,fp_on_hard_neg,recall_by_type
always_yes,0.35,1.0,0.52,78,10,"{'create_pay': 1.0, 'self_approve': 1.0, 'spli..."
always_no,0.0,0.0,0.0,0,0,"{'create_pay': 0.0, 'self_approve': 0.0, 'spli..."


In [10]:
rowlevel_df = df[df.violation_type != "split_po"]   # split_po can't be judged from one row
dev = (rowlevel_df.groupby("violation_type")
                  .sample(n=9, random_state=1)
                  .sort_values("po_id")
                  .copy())
test = df[~df.po_id.isin(dev.po_id)].copy()

print(len(dev), len(test))
print(dev.violation_type.value_counts())

27 93
violation_type
create_pay      9
self_approve    9
none            9
Name: count, dtype: int64


In [11]:
def run_variant(name, fn, models, rows):
    cache = Path(f"results/sod_{name}.jsonl")
    done = set()
    if cache.exists():
        done = {(j["model"], j["po_id"]) for j in map(json.loads, cache.read_text(encoding="utf-8").splitlines())}
    for model in models:
        fn(model, rows.iloc[0])  # warm-up, not logged
        for _, r in rows.iterrows():
            if (model, r.po_id) in done:
                continue
            o = fn(model, r)
            with cache.open("a", encoding="utf-8") as f:
                f.write(json.dumps(o) + "\n")
        subprocess.run(["ollama", "stop", model], capture_output=True)
    res = pd.DataFrame(map(json.loads, cache.read_text(encoding="utf-8").splitlines()))
    return rows.merge(res, on="po_id")

def score_variant(full_v, models):
    out = {}
    for m in models:
        d = full_v[full_v.model == m].copy()
        d["pred_viol"] = d.pred_viol.astype(bool)
        e = evaluate(d, "pred_viol")
        e["sec_per_row"] = round(d.secs.mean(), 1)
        out[m] = e
    return pd.DataFrame(out).T

In [12]:
def shot(po, c, a, p, vtype):
    txt = f"{po} | date 2026-01-01 | vendor: Example Vendor | amount: INR 10,000 | created by: {c} | approved by: {a} | payment processed by: {p}"
    return [{"role": "user", "content": txt},
            {"role": "assistant", "content": Verdict(violation=vtype != "none", type=vtype).model_dump_json()}]

SHOTS = (shot("PO-9001", "U0101", "U0102", "U0103", "none")
       + shot("PO-9002", "U0104", "U0105", "U0104", "create_pay")
       + shot("PO-9003", "U0106", "U0107", "U0108", "none")
       + shot("PO-9004", "U0109", "U0109", "U0110", "self_approve")
       + shot("PO-9005", "U0111", "U0112", "U0111", "create_pay")
       + shot("PO-9006", "U0113", "U0114", "U0115", "none"))

def classify_fewshot(model, r):
    resp = ollama.chat(model=model,
        messages=[{"role": "system", "content": SYSTEM}] + SHOTS + [{"role": "user", "content": row_text(r)}],
        format=Verdict.model_json_schema(),
        options={"temperature": 0, "num_ctx": 2048, "num_predict": 60})
    try:
        v = Verdict.model_validate_json(resp.message.content); ok, pv, pt = True, v.violation, v.type
    except Exception:
        ok, pv, pt = False, False, "none"
    return dict(model=model, po_id=r.po_id, parse_ok=ok, pred_viol=pv, pred_type=pt,
                secs=resp.total_duration / 1e9, raw=resp.message.content)

MODELS = ["qwen2.5:1.5b", "llama3.2:3b", "qwen2.5:3b"]
dev_b = run_variant("fewshot_dev", classify_fewshot, MODELS, dev)
display(score_variant(dev_b, MODELS))

,precision,recall,f1,fp_on_clean,fp_on_hard_neg,recall_by_type,sec_per_row
qwen2.5:1.5b,0.0,0.0,0.0,0,0,"{'create_pay': 0.0, 'self_approve': 0.0, 'spli...",3.3
llama3.2:3b,0.9,1.0,0.95,2,0,"{'create_pay': 1.0, 'self_approve': 1.0, 'spli...",6.9
qwen2.5:3b,1.0,0.17,0.29,0,0,"{'create_pay': 0.11, 'self_approve': 0.22, 'sp...",6.8


In [13]:
class Steps(BaseModel):
    created_by: str
    approved_by: str
    paid_by: str
    created_equals_approved: bool
    created_equals_paid: bool
    violation: bool
    type: Literal["none", "create_pay", "self_approve"]

SYSTEM_C = """You are an internal-controls auditor reviewing ONE purchase order.
Fill in the JSON fields in order:
1. Copy the three user IDs exactly as written.
2. created_equals_approved: true only if the created-by ID and the approved-by ID are the identical string.
3. created_equals_paid: true only if the created-by ID and the payment-processed-by ID are the identical string.
4. violation is true if either of those two is true. type is create_pay, self_approve, or none."""

def classify_steps(model, r):
    resp = ollama.chat(model=model,
        messages=[{"role": "system", "content": SYSTEM_C}, {"role": "user", "content": row_text(r)}],
        format=Steps.model_json_schema(),
        options={"temperature": 0, "num_ctx": 2048, "num_predict": 150})
    try:
        v = Steps.model_validate_json(resp.message.content)
        ok, pv, pt = True, v.violation, v.type
        extra = dict(ce_a=v.created_equals_approved, ce_p=v.created_equals_paid,
                     copied=[v.created_by, v.approved_by, v.paid_by])
    except Exception:
        ok, pv, pt, extra = False, False, "none", {}
    return dict(model=model, po_id=r.po_id, parse_ok=ok, pred_viol=pv, pred_type=pt,
                secs=resp.total_duration / 1e9, raw=resp.message.content, **extra)

dev_c = run_variant("steps_dev", classify_steps, MODELS, dev)
display(score_variant(dev_c, MODELS))

,precision,recall,f1,fp_on_clean,fp_on_hard_neg,recall_by_type,sec_per_row
qwen2.5:1.5b,0.67,1.0,0.8,9,2,"{'create_pay': 1.0, 'self_approve': 1.0, 'spli...",7.9
llama3.2:3b,1.0,0.5,0.67,0,0,"{'create_pay': 1.0, 'self_approve': 0.0, 'spli...",14.4
qwen2.5:3b,0.64,0.5,0.56,5,2,"{'create_pay': 0.0, 'self_approve': 1.0, 'spli...",15.6


In [14]:
dev_ref = dev.assign(always_yes=True, always_no=False)
display(pd.DataFrame({"always_yes": evaluate(dev_ref, "always_yes"),
                      "always_no": evaluate(dev_ref, "always_no")}).T)

b = dev_b[dev_b.model == "llama3.2:3b"].copy()
b["pred_viol"] = b.pred_viol.astype(bool)
print(confusion_matrix(b.violation, b.pred_viol))   # rows: truth no/yes, cols: pred no/yes
display(b[b.violation != b.pred_viol][["po_id", "created_by", "approved_by", "payment_processed_by",
                                      "amount", "violation_type", "pred_type"]])

,precision,recall,f1,fp_on_clean,fp_on_hard_neg,recall_by_type
always_yes,0.67,1.0,0.8,9,2,"{'create_pay': 1.0, 'self_approve': 1.0, 'spli..."
always_no,0.0,0.0,0.0,0,0,"{'create_pay': 0.0, 'self_approve': 0.0, 'spli..."


[[ 7  2]
 [ 0 18]]


,po_id,created_by,approved_by,payment_processed_by,amount,violation_type,pred_type
70,PO-0104,U5012,U7912,U1488,5670.0,none,create_pay
76,PO-0117,U4657,U2679,U1520,26910.0,none,create_pay


In [15]:
for m in ["llama3.2:3b", "qwen2.5:3b"]:
    d = dev_c[dev_c.model == m].dropna(subset=["ce_p"])
    print(m,
          "| copied IDs right:", round((d.copied == d[["created_by","approved_by","payment_processed_by"]].values.tolist()).mean(), 2),
          "| created==paid flag right:", round((d.ce_p == (d.created_by == d.payment_processed_by)).mean(), 2),
          "| created==approved flag right:", round((d.ce_a == (d.created_by == d.approved_by)).mean(), 2))

llama3.2:3b | copied IDs right: 0.93 | created==paid flag right: 1.0 | created==approved flag right: 0.93
qwen2.5:3b | copied IDs right: 1.0 | created==paid flag right: 0.67 | created==approved flag right: 1.0


In [16]:
test_res = run_variant("fewshot_test", classify_fewshot, ["llama3.2:3b"], test)
t = test_res.copy()
t["pred_viol"] = t.pred_viol.astype(bool)

row_level = t[t.violation_type != "split_po"]
print("all test rows:", evaluate(t, "pred_viol"))
print("row-level only (no split_po):", evaluate(row_level, "pred_viol"))
print(confusion_matrix(row_level.violation, row_level.pred_viol))

ref = row_level.assign(always_yes=True, always_no=False)
display(pd.DataFrame({"always_yes": evaluate(ref, "always_yes"),
                      "always_no": evaluate(ref, "always_no")}).T)

all test rows: {'precision': 0.33, 'recall': 0.26, 'f1': 0.29, 'fp_on_clean': 38, 'fp_on_hard_neg': 4, 'recall_by_type': {'create_pay': np.float64(0.41), 'self_approve': np.float64(0.44), 'split_po': np.float64(0.11)}}
row-level only (no split_po): {'precision': 0.28, 'recall': 0.42, 'f1': 0.34, 'fp_on_clean': 38, 'fp_on_hard_neg': 4, 'recall_by_type': {'create_pay': np.float64(0.41), 'self_approve': np.float64(0.44), 'split_po': nan}}
[[169  38]
 [ 21  15]]


,precision,recall,f1,fp_on_clean,fp_on_hard_neg,recall_by_type
always_yes,0.15,1.0,0.26,207,24,"{'create_pay': 1.0, 'self_approve': 1.0, 'spli..."
always_no,0.0,0.0,0.0,0,0,"{'create_pay': 0.0, 'self_approve': 0.0, 'spli..."


In [17]:
# 1) what kind of false alarm is it?
fp = row_level[(~row_level.violation) & row_level.pred_viol]
tn = row_level[(~row_level.violation) & (~row_level.pred_viol)]
print(fp.pred_type.value_counts())

# 2) do false alarms involve IDs that look alike?
from difflib import SequenceMatcher
sim = lambda a, b: SequenceMatcher(None, a, b).ratio()
for name, g in [("false alarms", fp), ("correct clean", tn)]:
    print(name,
          "| created~paid similarity:", round(g.apply(lambda r: sim(r.created_by, r.payment_processed_by), axis=1).mean(), 2),
          "| created~approved similarity:", round(g.apply(lambda r: sim(r.created_by, r.approved_by), axis=1).mean(), 2))

# 3) were the split_po hits just chance?
sp = t[t.violation_type == "split_po"]
print(int(sp.pred_viol.sum()), "of", len(sp), "split_po rows flagged; types:", sp.pred_type.value_counts().to_dict())

pred_type
create_pay      30
self_approve     8
Name: count, dtype: int64
false alarms | created~paid similarity: 0.46 | created~approved similarity: 0.46
correct clean | created~paid similarity: 0.45 | created~approved similarity: 0.41
4 of 36 split_po rows flagged; types: {'none': 32, 'create_pay': 4}


In [18]:
other = run_variant("fewshot_test", classify_fewshot, ["qwen2.5:3b", "qwen2.5:1.5b"], test)

In [19]:
other["pred_viol"] = other.pred_viol.astype(bool)
for m in ["llama3.2:3b", "qwen2.5:3b", "qwen2.5:1.5b"]:
    d = other[(other.model == m) & (other.violation_type != "split_po")]
    print(m, evaluate(d, "pred_viol"), confusion_matrix(d.violation, d.pred_viol).tolist())

llama3.2:3b {'precision': 0.24, 'recall': 1.0, 'f1': 0.39, 'fp_on_clean': 37, 'fp_on_hard_neg': 4, 'recall_by_type': {'create_pay': np.float64(1.0), 'self_approve': np.float64(1.0), 'split_po': nan}} [[32, 37], [0, 12]]
qwen2.5:3b {'precision': 1.0, 'recall': 0.25, 'f1': 0.4, 'fp_on_clean': 0, 'fp_on_hard_neg': 0, 'recall_by_type': {'create_pay': np.float64(0.22), 'self_approve': np.float64(0.33), 'split_po': nan}} [[69, 0], [9, 3]]
qwen2.5:1.5b {'precision': 0.0, 'recall': 0.0, 'f1': 0.0, 'fp_on_clean': 1, 'fp_on_hard_neg': 0, 'recall_by_type': {'create_pay': np.float64(0.0), 'self_approve': np.float64(0.0), 'split_po': nan}} [[68, 1], [12, 0]]


In [20]:
d = dev_c[dev_c.model == "llama3.2:3b"].dropna(subset=["ce_p"]).copy()
d["pred_viol_code"] = d.ce_a.astype(bool) | d.ce_p.astype(bool)
print(confusion_matrix(d.violation, d.pred_viol_code))
print(evaluate(d, "pred_viol_code"))

[[ 9  0]
 [ 0 18]]
{'precision': 1.0, 'recall': 1.0, 'f1': 1.0, 'fp_on_clean': 0, 'fp_on_hard_neg': 0, 'recall_by_type': {'create_pay': np.float64(1.0), 'self_approve': np.float64(1.0), 'split_po': nan}}


## Findings

**Setup.** 120 synthetic purchase orders with injected segregation-of-duties violations (create-and-pay, create-and-approve, and split POs just under an approval threshold), 35% violation rate. Models ran locally via Ollama on a CPU-only Ryzen 5 2500U laptop with 8 GB RAM. 27 row-level rows were used as a development set for prompt iteration; the other 93 (81 row-level, 12 split-PO) were held out, with prompts frozen on dev before they were run on them.

### Results

**1. Zero-shot (all 120 rows).** All three models behaved like constant classifiers. qwen2.5:1.5b labelled every row `self_approve` (precision 0.35, recall 1.0, F1 0.52, which equals an always-"violation" classifier at this violation rate). llama3.2:3b and qwen2.5:3b flagged nothing (recall 0).

**2. Prompt variants (dev set, 27 rows: 9 clean, 9 create_pay, 9 self_approve; always-"violation" reference F1 = 0.80).**

| model | few-shot: precision / recall / F1 | step-by-step: precision / recall / F1 |
|---|---|---|
| qwen2.5:1.5b | 0.00 / 0.00 / 0.00 (flags nothing) | 0.67 / 1.00 / 0.80 (flags everything) |
| llama3.2:3b | 0.90 / 1.00 / 0.95 | 1.00 / 0.50 / 0.67 |
| qwen2.5:3b | 1.00 / 0.17 / 0.29 | 0.64 / 0.50 / 0.56 |

Each model's behaviour changed sharply with the prompt format, and no model was good under both.

**3. Held-out test (few-shot prompt frozen on dev; row-level rows only: 12 violations, 69 clean).**

| model | TP | FP | FN | TN | precision | recall | F1 |
|---|---|---|---|---|---|---|---|
| llama3.2:3b | 12 | 37 | 0 | 32 | 0.24 | 1.00 | 0.39 |
| qwen2.5:3b | 3 | 0 | 9 | 69 | 1.00 | 0.25 | 0.40 |
| qwen2.5:1.5b | 0 | 1 | 12 | 68 | 0.00 | 0.00 | 0.00 |
| always "violation" | 12 | 69 | 0 | 0 | 0.15 | 1.00 | 0.26 |
| always "no violation" | 0 | 0 | 12 | 69 | 0.00 | 0.00 | 0.00 |

llama3.2:3b and qwen2.5:3b have nearly equal F1 but opposite failure modes: llama flags 49 of 81 rows (catching every violation, with 37 false alarms), while qwen2.5:3b flags 3 of 81 (no false alarms, 9 of 12 violations missed). qwen2.5:3b and qwen2.5:1.5b behaved on test as they did on dev (almost never flagging). llama3.2:3b's dev result did not hold up: 2 false alarms in 9 clean dev rows became 37 in 69 on test. I attribute this to the small dev set and to choosing the best of several configurations on it. With only 12 violations in the test set, recall estimates are coarse. None of the three models is usable as a detector in this setup.

**4. Where llama3.2:3b failed on test.** 30 of its 37 false alarms were labelled `create_pay` and 7 `self_approve`. ID string similarity did not differ between false alarms and correct clean rows (created~paid 0.46 vs 0.45), so I found no support for a "look-alike IDs" explanation. It flagged 4 of the 12 split-PO rows, all as `create_pay`, which is the wrong reason; a single-row prompt cannot see a multi-row pattern, so these are chance hits, not detections.

**5. Diagnosing the step-by-step variant (dev).** The prompt asked the model to copy the three IDs, answer two equality questions, then give a verdict.
- llama3.2:3b copied IDs correctly 93% of the time and answered the two equality questions correctly (created==paid 100%, created==approved 93%), but missed every `self_approve` row at the verdict stage. Its comparison of IDs was fine; it did not combine the flags into a verdict.
- qwen2.5:3b copied IDs correctly (100%) but its created==paid flag was right only 67%, no better than always answering "false" on this dev set. This matches its zero `create_pay` recall.
- Deciding the verdict in code from llama3.2:3b's two flags (violation = created==approved OR created==paid) gave 27/27 correct on dev (TP 18, TN 9). This was measured on the same rows used to design the prompt, so it is a hypothesis about where the model fails, not a performance estimate. The step-by-step prompt takes about 14 s per row, and an exact comparison in pandas gives the same answer instantly, so this is not a better detector than the rule.

**6. Rule baseline.** A pandas rule check scores perfectly on all 120 rows. This is expected because the labels were generated from the same rules, so it is a sanity check on the code and not an achievement.

### Conclusion

For exact-match controls like these, a deterministic rule is the right tool. In my experiments small local models did not reliably do the matching in a single step, and where they failed depended on the model and the prompt: llama3.2:3b compared IDs well but did not turn its own answers into a verdict, while qwen2.5:3b did not get the created-vs-paid comparison right. The split-PO pattern needs several rows at once, so a single-row prompt cannot detect it. This suggests a design where code and tools do exact checks and the model handles what rules cannot, which I have not tested.

### Limitations

- Synthetic data generated with the same rules as the baseline; unrealistically high violation rate (35%).
- Clean POs draw creator, approver and payer IDs from separate pools, while violations break that pattern, which may allow shortcuts; only 14 distinct user IDs.
- Small samples: 27 dev rows (9 clean) and 81 row-level test rows (12 violations), so rates are imprecise and the dev-to-test change for llama3.2:3b shows how unreliable small dev sets are.
- One prompt wording per variant, temperature 0, one run per row.
- The test rows were not regenerated with a second independent set, and they were run once per model with a prompt frozen on dev.
- Only llama3.2:3b was diagnosed in detail on the step-by-step variant; the code-verdict result is dev-only.
- No fine-tuning was done.
- `fp_on_clean` in the tables includes the near-threshold clean rows (`fp_on_hard_neg` is a subset of it).